# 01 — Data understanding and cleaning notes

**Phase 3 of Energy Intelligence.** No model is trained here.

## What are we doing?

We inspect the raw Spanish hourly load file before any feature engineering.

## Why?

Charts and models are misleading if timestamps are wrong, hours are missing, or “outliers” are actually winter peaks. Cleaning rules should come from this inspection, not from habit.

## Setup

In [1]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_rows", 20)
plt.rcParams.update(
    {
        "figure.figsize": (10, 4.2),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
    }
)
BRAND = "#0f766e"


def project_root() -> Path:
    here = Path.cwd()
    for candidate in (here, here.parent):
        if (candidate / "data" / "raw" / "spain_hourly_load.csv").exists():
            return candidate
    raise FileNotFoundError(
        "spain_hourly_load.csv not found. From the project root run: "
        "python scripts/download_dataset.py"
    )


ROOT = project_root()
DATA_PATH = ROOT / "data" / "raw" / "spain_hourly_load.csv"
print(f"Using {DATA_PATH}")

Using C:\Users\Hammad Ali\Energy Intelligence\data\raw\spain_hourly_load.csv


## Load the raw file

The CSV has three columns created in Phase 2. `load_mw` is national electricity demand in Spain.

In [2]:
raw = pd.read_csv(DATA_PATH)
raw.head()

,utc_timestamp,cet_cest_timestamp,load_mw
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,NaN
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,22734.0
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,21286.0
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,20264.0


In [3]:
print("rows, columns:", raw.shape)
print()
print(raw.dtypes)
print()
print("null counts:")
print(raw.isna().sum())

rows, columns: (50401, 3)

utc_timestamp             str
cet_cest_timestamp        str
load_mw               float64
dtype: object

null counts:
utc_timestamp          0
cet_cest_timestamp     0
load_mw               23
dtype: int64


### What this means

Pandas already read `load_mw` as a number. Empty cells became `NaN` (not zero). Treating missing demand as 0 MW would be a serious error: it would look like the grid shut off.

## Parse timestamps

In [4]:
df = raw.copy()
df["utc"] = pd.to_datetime(df["utc_timestamp"], utc=True)
# Local clock in Spain. Demand follows this, not UTC.
df["local"] = df["utc"].dt.tz_convert("Europe/Madrid")
df = df.sort_values("utc").reset_index(drop=True)
df[["utc_timestamp", "cet_cest_timestamp", "utc", "local", "load_mw"]].head(4)

,utc_timestamp,cet_cest_timestamp,utc,local,load_mw
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,2014-12-31 23:00:00+00:00,2015-01-01 00:00:00+01:00,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,2015-01-01 00:00:00+00:00,2015-01-01 01:00:00+01:00,NaN
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,2015-01-01 01:00:00+00:00,2015-01-01 02:00:00+01:00,22734.0
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,2015-01-01 02:00:00+00:00,2015-01-01 03:00:00+01:00,21286.0


In [5]:
print("UTC range:", df["utc"].min(), "->", df["utc"].max())
print("Duplicate UTC timestamps:", int(df["utc"].duplicated().sum()))
print("Already sorted by UTC:", bool(df["utc"].is_monotonic_increasing))

UTC range: 2014-12-31 23:00:00+00:00 -> 2020-09-30 23:00:00+00:00
Duplicate UTC timestamps: 0
Already sorted by UTC: True


## Is the clock regular?

For hourly forecasting we need one row per hour. Gaps would break lag features (`lag_1`, `lag_24`).

In [6]:
gaps = df["utc"].diff().dropna()
print(gaps.value_counts())
print()
print("Gaps longer than 1 hour:", int((gaps > pd.Timedelta(hours=1)).sum()))
print("Steps shorter than 1 hour:", int((gaps < pd.Timedelta(hours=1)).sum()))

utc
0 days 01:00:00    50400
Name: count, dtype: int64

Gaps longer than 1 hour: 0
Steps shorter than 1 hour: 0


### What this means

The UTC index is a complete hourly grid: no duplicates, no missing hours, no extra half-hours. Daylight-saving changes in Spain do **not** punch holes in UTC. That is why we keep UTC as the index and convert to `Europe/Madrid` only for hour-of-day plots.

**What could go wrong:** if we grouped by the raw `cet_cest_timestamp` string and parsed mixed `+0100` / `+0200` offsets without care, pandas can raise a mixed-timezone error. Convert from UTC instead.

## Missing demand values

In [7]:
missing = df.loc[df["load_mw"].isna(), ["utc", "local", "load_mw"]]
print(f"Missing load_mw: {len(missing)} of {len(df)} rows "
      f"({100 * len(missing) / len(df):.3f}%)")
missing

Missing load_mw: 23 of 50401 rows (0.046%)


,utc,local,load_mw
0,2014-12-31 23:00:00+00:00,2015-01-01 00:00:00+01:00,NaN
1,2015-01-01 00:00:00+00:00,2015-01-01 01:00:00+01:00,NaN
31,2015-01-02 06:00:00+00:00,2015-01-02 07:00:00+01:00,NaN
32,2015-01-02 07:00:00+00:00,2015-01-02 08:00:00+01:00,NaN
33,2015-01-02 08:00:00+00:00,2015-01-02 09:00:00+01:00,NaN
...,...,...,...
2896,2015-05-01 15:00:00+00:00,2015-05-01 17:00:00+02:00,NaN
50336,2020-09-28 07:00:00+00:00,2020-09-28 09:00:00+02:00,NaN
50337,2020-09-28 08:00:00+00:00,2020-09-28 10:00:00+02:00,NaN
50338,2020-09-28 09:00:00+00:00,2020-09-28 11:00:00+02:00,NaN


### What this means

Missingness is rare (~0.05%) and clustered:

- the first hours of 2015 (series start)
- several hours on 2 January 2015
- six hours on 1 May 2015 (a public holiday in Spain — a reporting gap is plausible)
- a few hours in late September 2020, plus the last row

This is **not** random salt-and-pepper noise. Short clusters can be interpolated; long leading/trailing gaps should not be invented.

### Cleaning rule to use in Phase 4

1. Keep UTC as the index.
2. Interpolate `load_mw` in time only for gaps of **up to 3 hours**.
3. Leave longer gaps as missing and drop those hours when building lag features.
4. Never fill missing demand with 0.

In [8]:
load = df.set_index("utc")["load_mw"]
interpolated = load.interpolate(method="time", limit=3)
still_missing = interpolated.isna().sum()
print(f"Missing after time-interpolate (limit=3 hours): {still_missing}")
print("We do not save a processed file yet. Phase 4 will freeze this rule in a Python module.")

Missing after time-interpolate (limit=3 hours): 10
We do not save a processed file yet. Phase 4 will freeze this rule in a Python module.


## Duplicates and invalid values

In [9]:
print("Duplicate full rows:", int(df.duplicated().sum()))
print("load_mw <= 0:", int((df["load_mw"] <= 0).sum()))
print("Non-numeric leftovers:", int(pd.to_numeric(raw["load_mw"], errors="coerce").isna().sum() - df["load_mw"].isna().sum()))

Duplicate full rows: 0


load_mw <= 0: 0
Non-numeric leftovers: 0


### What this means

No duplicate rows, and no zero or negative demand. Invalid records are the 23 `NaN`s already listed.

## Scale of demand

In [10]:
df["load_mw"].describe().round(1)

count    50378.0
mean     28441.6
std       4626.1
min      16575.0
25%      24581.0
50%      28522.0
75%      31990.0
max      41015.0
Name: load_mw, dtype: float64

National Spanish load sits roughly between **16.6 GW and 41.0 GW**, with a mean near **28.4 GW**. That range is physically plausible for a country of Spain’s size. We will look at *when* the extremes happen in notebook 02, before calling them errors.

## Cleaning summary (input to Phase 4)

| Check | Result | Action later |
| --- | --- | --- |
| Timezone | Use UTC index; local time via `Europe/Madrid` | Implement in preprocessing |
| Sorting | Already chronological | Still sort defensively |
| Duplicates | None | Drop if they ever appear |
| Hourly gaps | None in UTC | `asfreq("h")` as a safety net |
| Missing load | 23 clustered `NaN`s | Interpolate ≤ 3 hours; otherwise drop at feature time |
| Zeros / negatives | None | Reject if they appear |
| Outliers | See notebook 02 | Do not clip winter peaks without evidence they are errors |

Next: `02_eda.ipynb` for daily, weekly, and seasonal patterns.